# CLIMADA

In [1]:
import sim_mel as sm
from climada.hazard import Hazard, Centroids, TCTracks, TropCyclone
from climada.entity import Exposures
from climada.entity.exposures import LitPop
from climada.entity.impact_funcs import ImpactFuncSet, ImpfTropCyclone
# from climada.entity import Measure, MeasureSet, Entity
from climada.engine import ImpactCalc, Impact
# import os
# from climada.util import save, load
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import math
import collections
import climada.util.coordinates as u_coord
# import climada.entity.exposures.litpop as lp
from datetime import datetime as dt
import geopandas as gpd
from scipy import sparse as sp

import warnings
warnings.filterwarnings('ignore')

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(


In [2]:
## set parameters
# Exposure
my_country = 'AUS'
m, n = 1,1 # based on analysis by Eberenz et al. (2020), (1,1) is the best combination.
my_fin_mode = 'pc'
ref_year = 2023

# Hazard
start_year = 1980
end_year = 2023
n_synth_tracks = 300

my_res_arcsec = 600
n_sim = 1000000 # 1million
starting_phase = 'Nino'

## Exposure

In [3]:
# Exposure
exp = LitPop.from_countries(my_country, res_arcsec=my_res_arcsec, reference_year=ref_year, exponents =(m,n), fin_mode=my_fin_mode)
exp.check()

2025-02-06 09:35:33,034 - climada.entity.exposures.litpop.gpw_population - WARNING - Reference year: 2023. Using nearest available year for GPW data: 2020
2025-02-06 09:35:33,040 - climada.entity.exposures.litpop.gpw_population - WARNING - Reference year: 2023. Using nearest available year for GPW data: 2020
2025-02-06 09:35:33,047 - climada.entity.exposures.litpop.gpw_population - WARNING - Reference year: 2023. Using nearest available year for GPW data: 2020
2025-02-06 09:35:33,089 - climada.entity.exposures.litpop.gpw_population - WARNING - Reference year: 2023. Using nearest available year for GPW data: 2020
2025-02-06 09:35:33,107 - climada.entity.exposures.litpop.gpw_population - WARNING - Reference year: 2023. Using nearest available year for GPW data: 2020
2025-02-06 09:35:35,599 - climada.entity.exposures.litpop.gpw_population - WARNING - Reference year: 2023. Using nearest available year for GPW data: 2020
2025-02-06 09:35:35,612 - climada.entity.exposures.litpop.gpw_populati

In [4]:
print(f'Total exposure value to {my_country}: USD {exp.gdf.value.sum():,.2f} / AUD {exp.gdf.value.sum()/0.6630:,.2f} ({ref_year})')

Total exposure value to AUS: USD 6,547,447,959,025.30 / AUD 9,875,487,117,685.22 (2023)


## Hazard

In [ ]:
# Hazard
haz = TropCyclone.from_hdf5('../Hazards/haz_aus_300synth_decay.hdf5')
haz.check()
haz.centroids.set_region_id()
haz.centroids.set_on_land()
haz.centroids.set_lat_lon_to_meta()
exp.assign_centroids(haz)

## Impact

In [6]:
impf_tc = ImpfTropCyclone.from_emanuel_usa()

impf_set = ImpactFuncSet([impf_tc])
impf_set.check()

## Results
### 1-year results

### Done

In [8]:
loss_orig = sm.sim_losses_per_point(haz, exp, impf_set, 'orig', n_sim, n_years=1, num_cpus=8)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_orig.npz', loss_orig)

If fqcy = 'orig' then loc, intensity, and damage are not used. They are all set to False.


/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

In [9]:
loss_MMNHPP_FFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = False, intensity = False, damage = False, num_cpus=8)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_FFF.npz', loss_MMNHPP_FFF)

In [10]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, intensity = True, damage = True, num_cpus=8)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_TTT_50.npz', loss_MMNHPP_TTT)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

In [11]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, p_loc = 0.8, intensity = True, damage = True, num_cpus=8)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_TTT_80.npz', loss_MMNHPP_TTT)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

In [12]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, p_loc = 1, intensity = True, damage = True, num_cpus=8)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_TTT_100.npz', loss_MMNHPP_TTT)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

In [13]:
loss_MMNHPP_FTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = False, intensity = True, damage = False)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_FTF.npz', loss_MMNHPP_FTF)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

In [14]:
loss_MMNHPP_FFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = False, intensity = False, damage = True)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_FFT.npz', loss_MMNHPP_FFT)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

In [15]:
loss_MMNHPP_TFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_years=1, loc = True, intensity = False, damage = False)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_TFF_50.npz', loss_MMNHPP_TFF)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

### Not done

In [ ]:
loss_MMNHPP_TTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, loc = True, intensity = True, damage = False)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_TTF.npz', loss_MMNHPP_TTF)

In [ ]:
loss_MMNHPP_TFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, loc = True, intensity = False, damage = True)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_TFT.npz', loss_MMNHPP_TFT)

In [ ]:
loss_MMNHPP_FTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, loc = False, intensity = True, damage = True)
sp.save_npz('AUS outputs/Loss outputs 1yr decay/loss_MMNHPP_FTT.npz', loss_MMNHPP_FTT)

### 5-year losses

In [7]:
n_sim = 1000000
n_year = 5
seed = 23

### Done

In [9]:
loss_orig = sm.sim_losses_per_point(haz, exp, impf_set, 'orig', n_sim, n_year)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_orig.npz', loss_orig)

If fqcy = 'orig' then loc, intensity, and damage are not used. They are all set to False.


/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

In [10]:
loss_MMNHPP_FFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = False, intensity = False, damage = False, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_FFF.npz', loss_MMNHPP_FFF)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

### Not done

In [8]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = True, intensity = True, damage = True, seed = seed, num_cpus=8)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_TTT_50.npz', loss_MMNHPP_TTT)

/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider upgrading.
  warnings.warn(
/Users/melissarenard/miniforge3/envs/climada_env/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 12.0.1. Please consider 

KeyboardInterrupt: 

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = True, p_loc = 0.8, intensity = True, damage = True, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_TTT_80.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_TTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = True, p_loc = 1, intensity = True, damage = True, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_TTT_100.npz', loss_MMNHPP_TTT)

In [ ]:
loss_MMNHPP_TFF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = True, intensity = False, damage = False, seed=seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_TFF.npz', loss_MMNHPP_TFF)

In [ ]:
loss_MMNHPP_FTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = False, intensity = True, damage = False, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_FTF.npz', loss_MMNHPP_FTF)

In [ ]:
loss_MMNHPP_FFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = False, intensity = False, damage = True, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_FFT.npz', loss_MMNHPP_FFT)

In [ ]:
loss_MMNHPP_TTF = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = True, intensity = True, damage = False, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_TTF.npz', loss_MMNHPP_TTF)

In [ ]:
loss_MMNHPP_TFT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = True, intensity = False, damage = True, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_TFT.npz', loss_MMNHPP_TFT)

In [ ]:
loss_MMNHPP_FTT = sm.sim_losses_per_point(haz, exp, impf_set, 'MMNHPP', n_sim, n_year, loc = False, intensity = True, damage = True, seed = seed)
sp.save_npz('AUS outputs/Loss outputs 5yr decay/loss_MMNHPP_FTT.npz', loss_MMNHPP_FTT)